# Fine-Tuning Results (Thesis Section 4.4.3)

Compares zero-shot vs fine-tuned Gemma 2 9B-IT on the test set (510 samples).

**Sections:**
1. Load Results
2. Overall Before/After Comparison
3. Per-Rubric Improvement
4. Off-Topic Detection Improvement
5. Grade Distribution Shift
6. Statistical Tests (Paired t-test, Wilcoxon, TOST)
7. Per-Band Accuracy Comparison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import cohen_kappa_score, confusion_matrix

sns.set_theme(style='whitegrid', font_scale=1.1)

BASE = '../Finetuning'
RUBRICS = ['Clarity', 'Terminology', 'Coverage', 'Accuracy']

## 1. Load Results

In [ ]:
zs = pd.read_csv(f'{BASE}/zeroshot_finetuned_prompt_results.csv')
ft = pd.read_csv(f'{BASE}/finetuned_results.csv')

# Normalize column names
for df in [zs, ft]:
    col_map = {}
    for c in df.columns:
        if c in ('Within_1', 'Within_1.0'): col_map[c] = 'Within_1'
        elif c in ('Within_05', 'Within_0.5'): col_map[c] = 'Within_05'
    df.rename(columns=col_map, inplace=True)

print(f'Zero-shot: {len(zs)} samples')
print(f'Fine-tuned: {len(ft)} samples')

## 2. Overall Before/After Comparison

In [ ]:
def compute_metrics(df):
    diff = np.abs(df['LLM_Final'].values - df['Human_Final'].values)
    acc_1 = np.mean(diff <= 1.0) * 100
    acc_05 = np.mean(diff <= 0.5) * 100
    mae = np.mean(diff)

    h_total = (df['Human_Clarity'] + df['Human_Terminology'] + df['Human_Coverage'] + df['Human_Accuracy']).astype(int)
    l_total = (df['LLM_Clarity'] + df['LLM_Terminology'] + df['LLM_Coverage'] + df['LLM_Accuracy']).astype(int).clip(0, 10)
    qwk = cohen_kappa_score(h_total, l_total, weights='quadratic')

    # Off-topic
    if 'OffTopic_Match' in df.columns:
        ot_acc = df['OffTopic_Match'].astype(bool).mean() * 100
    else:
        ot_acc = np.nan

    return {'Acc ±1.0 (%)': acc_1, 'Acc ±0.5 (%)': acc_05, 'MAE': mae, 'QWK': qwk, 'OT Acc (%)': ot_acc}


zs_m = compute_metrics(zs)
ft_m = compute_metrics(ft)

comp_df = pd.DataFrame([
    {'Model': 'Zero-Shot', **{k: round(v, 3) for k, v in zs_m.items()}},
    {'Model': 'Fine-Tuned', **{k: round(v, 3) for k, v in ft_m.items()}},
    {'Model': 'Improvement', **{k: round(ft_m[k] - zs_m[k], 3) for k in zs_m}}
])
print('Overall Comparison (510 test samples):')
comp_df

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
metric_keys = ['Acc ±1.0 (%)', 'Acc ±0.5 (%)', 'MAE', 'QWK']

for ax, metric in zip(axes, metric_keys):
    vals = [zs_m[metric], ft_m[metric]]
    colors = ['#e74c3c', '#2ecc71']
    bars = ax.bar(['Zero-Shot', 'Fine-Tuned'], vals, color=colors, edgecolor='white')
    ax.set_title(metric, fontsize=13, fontweight='bold')
    for bar, val in zip(bars, vals):
        fmt = f'{val:.1f}%' if '%' in metric else f'{val:.3f}'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                fmt, ha='center', va='bottom', fontsize=11, fontweight='bold')

fig.suptitle('Zero-Shot vs Fine-Tuned Performance', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Per-Rubric Improvement

In [ ]:
def per_rubric_metrics(df):
    results = {}
    for rubric in RUBRICS:
        h = df[f'Human_{rubric}'].values
        l = df[f'LLM_{rubric}'].values
        exact = np.mean(h == l) * 100
        within_1 = np.mean(np.abs(h - l) <= 1) * 100
        mae = np.mean(np.abs(h - l))
        max_score = 4 if rubric == 'Accuracy' else 2
        h_int = h.astype(int)
        l_int = np.clip(l.astype(int), 0, max_score)
        qwk = cohen_kappa_score(h_int, l_int, weights='quadratic')
        results[rubric] = {'Exact (%)': exact, '±1 (%)': within_1, 'MAE': mae, 'QWK': qwk}
    return pd.DataFrame(results).T


zs_rubric = per_rubric_metrics(zs)
ft_rubric = per_rubric_metrics(ft)
improvement = ft_rubric - zs_rubric

print('Zero-Shot per-rubric:')
display(zs_rubric.round(3))
print('\nFine-Tuned per-rubric:')
display(ft_rubric.round(3))
print('\nImprovement (FT - ZS):')
display(improvement.round(3))

## 4. Off-Topic Detection Improvement

In [ ]:
def off_topic_metrics(df):
    human_ot = df['Human_OffTopic'].isin(['Off-Topic', 'No Answer'])
    llm_ot = df['LLM_OffTopic'].isin(['Off-Topic', 'No Answer'])

    tp = (human_ot & llm_ot).sum()
    fp = (~human_ot & llm_ot).sum()
    fn = (human_ot & ~llm_ot).sum()
    tn = (~human_ot & ~llm_ot).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    acc = (tp + tn) / len(df) * 100

    return {'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
            'Precision': precision, 'Recall': recall, 'F1': f1, 'Accuracy (%)': acc}


zs_ot = off_topic_metrics(zs)
ft_ot = off_topic_metrics(ft)

ot_comp = pd.DataFrame([
    {'Model': 'Zero-Shot', **{k: round(v, 3) for k, v in zs_ot.items()}},
    {'Model': 'Fine-Tuned', **{k: round(v, 3) for k, v in ft_ot.items()}}
])
print('Off-Topic Detection Performance:')
ot_comp

## 5. Grade Distribution Shift

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
bins = np.arange(-0.25, 5.5, 0.5)

# Human
axes[0].hist(ft['Human_Final'], bins=bins, color='#2c3e50', edgecolor='white', alpha=0.85)
axes[0].set_title('Human Grades', fontsize=12)
axes[0].set_xlabel('Final Grade (0-5)')
axes[0].set_ylabel('Frequency')

# Zero-shot
axes[1].hist(zs['LLM_Final'], bins=bins, color='#e74c3c', edgecolor='white', alpha=0.85)
axes[1].set_title('Zero-Shot Grades', fontsize=12)
axes[1].set_xlabel('Final Grade (0-5)')

# Fine-tuned
axes[2].hist(ft['LLM_Final'], bins=bins, color='#2ecc71', edgecolor='white', alpha=0.85)
axes[2].set_title('Fine-Tuned Grades', fontsize=12)
axes[2].set_xlabel('Final Grade (0-5)')

fig.suptitle('Grade Distribution Comparison (510 test samples)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Overlaid distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(ft['Human_Final'], bins=bins, alpha=0.5, label='Human', color='#2c3e50', edgecolor='white')
ax.hist(zs['LLM_Final'], bins=bins, alpha=0.5, label='Zero-Shot', color='#e74c3c', edgecolor='white')
ax.hist(ft['LLM_Final'], bins=bins, alpha=0.5, label='Fine-Tuned', color='#2ecc71', edgecolor='white')
ax.set_xlabel('Final Grade (0-5)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Grade Distribution: Human vs Zero-Shot vs Fine-Tuned', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 6. Statistical Tests

In [ ]:
# Paired comparisons: fine-tuned grades vs human grades
human_grades = ft['Human_Final'].values
ft_grades = ft['LLM_Final'].values
diff = ft_grades - human_grades

print('Fine-Tuned vs Human Grade Statistical Tests')
print('='*55)

# 1. Paired t-test
t_stat, t_p = stats.ttest_rel(ft_grades, human_grades)
print(f'\n1. Paired t-test:')
print(f'   Mean diff: {np.mean(diff):.4f}')
print(f'   t = {t_stat:.4f}, p = {t_p:.4f}')
print(f'   Result: {"Significant" if t_p < 0.05 else "Not significant"} (alpha=0.05)')

# 2. Wilcoxon signed-rank test
w_stat, w_p = stats.wilcoxon(ft_grades, human_grades)
print(f'\n2. Wilcoxon signed-rank test:')
print(f'   W = {w_stat:.1f}, p = {w_p:.4f}')
print(f'   Result: {"Significant" if w_p < 0.05 else "Not significant"} (alpha=0.05)')

In [ ]:
# 3. TOST Equivalence Test (±0.25 margin on 0-5 scale)
margin = 0.25
n = len(diff)
mean_diff = np.mean(diff)
se_diff = np.std(diff, ddof=1) / np.sqrt(n)

# Upper bound test: H0: mean_diff >= margin
t_upper = (mean_diff - margin) / se_diff
p_upper = stats.t.cdf(t_upper, df=n-1)

# Lower bound test: H0: mean_diff <= -margin
t_lower = (mean_diff + margin) / se_diff
p_lower = 1 - stats.t.cdf(t_lower, df=n-1)

p_tost = max(p_upper, p_lower)

print(f'3. TOST Equivalence Test (margin = ±{margin}):')
print(f'   Mean diff: {mean_diff:.4f}, SE: {se_diff:.4f}')
print(f'   Upper bound: t = {t_upper:.4f}, p = {p_upper:.4f}')
print(f'   Lower bound: t = {t_lower:.4f}, p = {p_lower:.4f}')
print(f'   TOST p-value: {p_tost:.4f}')
print(f'   Result: {"Equivalent" if p_tost < 0.05 else "Not equivalent"} within ±{margin} (alpha=0.05)')
print(f'\n   90% CI for mean difference: [{mean_diff - 1.645*se_diff:.4f}, {mean_diff + 1.645*se_diff:.4f}]')

In [ ]:
# Visualization of difference distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of differences
ax = axes[0]
ax.hist(diff, bins=30, color='#3498db', edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', linestyle='--', linewidth=1.5, label='Zero')
ax.axvline(mean_diff, color='red', linestyle='-', linewidth=2, label=f'Mean={mean_diff:.3f}')
ax.axvline(margin, color='green', linestyle=':', linewidth=1.5, label=f'±{margin} margin')
ax.axvline(-margin, color='green', linestyle=':', linewidth=1.5)
ax.set_xlabel('Fine-Tuned - Human Grade', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Distribution of Grade Differences', fontsize=12)
ax.legend(fontsize=9)

# Bland-Altman plot
ax = axes[1]
mean_grades = (human_grades + ft_grades) / 2
ax.scatter(mean_grades, diff, alpha=0.3, s=20, color='#3498db')
ax.axhline(mean_diff, color='red', linestyle='-', linewidth=1.5, label=f'Mean={mean_diff:.3f}')
sd_diff = np.std(diff)
ax.axhline(mean_diff + 1.96*sd_diff, color='gray', linestyle='--', alpha=0.7, label='±1.96 SD')
ax.axhline(mean_diff - 1.96*sd_diff, color='gray', linestyle='--', alpha=0.7)
ax.set_xlabel('Mean of Human and FT Grades', fontsize=11)
ax.set_ylabel('Difference (FT - Human)', fontsize=11)
ax.set_title('Bland-Altman Plot', fontsize=12)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 7. Per-Band Accuracy Comparison

In [ ]:
def per_band_accuracy(df, tolerance=1.0):
    bands = sorted(df['Human_Final'].unique())
    results = {}
    for band in bands:
        mask = df['Human_Final'] == band
        if mask.sum() >= 5:
            d = np.abs(df.loc[mask, 'LLM_Final'] - df.loc[mask, 'Human_Final'])
            results[band] = np.mean(d <= tolerance) * 100
    return results


zs_bands = per_band_accuracy(zs)
ft_bands = per_band_accuracy(ft)

all_bands = sorted(set(zs_bands.keys()) | set(ft_bands.keys()))

band_comp = pd.DataFrame({
    'Human Grade': all_bands,
    'Zero-Shot (%)': [round(zs_bands.get(b, 0), 1) for b in all_bands],
    'Fine-Tuned (%)': [round(ft_bands.get(b, 0), 1) for b in all_bands],
    'Count': [int((ft['Human_Final'] == b).sum()) for b in all_bands]
})
band_comp['Improvement'] = band_comp['Fine-Tuned (%)'] - band_comp['Zero-Shot (%)']
print('Per-Band ±1.0 Accuracy:')
band_comp

In [ ]:
# Side-by-side bar chart
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(all_bands))
width = 0.35

bars1 = ax.bar(x - width/2, [zs_bands.get(b, 0) for b in all_bands], width,
               label='Zero-Shot', color='#e74c3c', edgecolor='white')
bars2 = ax.bar(x + width/2, [ft_bands.get(b, 0) for b in all_bands], width,
               label='Fine-Tuned', color='#2ecc71', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels([str(b) for b in all_bands])
ax.set_xlabel('Human Grade Band', fontsize=12)
ax.set_ylabel('±1.0 Accuracy (%)', fontsize=12)
ax.set_title('Per-Band Accuracy: Zero-Shot vs Fine-Tuned', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()